# Demo 05 - Sign-in activity heatmap (hour x weekday)

**Fast** (single table, aggregated) · **Pool:** Small · **Visual:** 2-D heatmap

**The question:** when does this organisation actually work, and what happens outside those
hours?

Every sign-in is bucketed by which day of the week and which hour of the day it landed in,
then the result is drawn as a grid with colour showing volume. The working week appears as
a solid block, and anything outside it stands on its own.

KQL can do the bucketing perfectly well. What it cannot do is draw the grid, and this is a
finding you see rather than read.

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Settings you can change

Two knobs, and they are the only things you should need to edit:

- `WORKSPACE` - which Log Analytics workspace to read from. Run
  `data_provider.list_databases()` if you are not sure of the name.
- `LOOKBACK_DAYS` - how far back to look. 14 days is enough to show a weekly rhythm
  without waiting around.

In [ ]:
WORKSPACE = "your-workspace-name"   # <-- Zava data lake workspace (run data_provider.list_databases())
LOOKBACK_DAYS = 14

## 3. Count sign-ins by hour and weekday

The expensive work happens in Spark, close to the data: read `SigninLogs`, keep the last
14 days, then count how many sign-ins landed in each (weekday, hour) bucket.

That collapses potentially millions of rows down to at most 168 (7 days x 24 hours), which
is small enough to pull back to the driver with `.toPandas()` and hand to a charting
library.

Aggregate first, collect second. Doing it the other way round is how notebooks fall over.

In [ ]:
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt

df = data_provider.read_table("SigninLogs", WORKSPACE)
df = df.filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS"))

# Aggregate in Spark (cheap) -> tiny 7x24 result
agg = (df.withColumn("weekday", F.date_format("TimeGenerated", "E"))
         .withColumn("hour", F.hour("TimeGenerated"))
         .groupBy("weekday", "hour").agg(F.count("*").alias("signins")))
pdf = agg.toPandas()
print("rows:", len(pdf))

## 4. Draw the heatmap

`pivot` reshapes those 168 rows into a 7 x 24 grid - weekdays down the side, hours across
the top - and Seaborn colours each cell by sign-in volume. Darker means busier.

**What to look for:** a solid block covering weekday working hours is your baseline. The
interesting cells are the isolated dark ones outside it. A burst of activity at 03:00 on a
Sunday is not somebody keeping odd hours, it is worth a question.

In [ ]:
if pdf.empty:
    print(f"No sign-ins in the last {LOOKBACK_DAYS} days - raise LOOKBACK_DAYS and re-run.")
else:
    order = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
    pivot = (pdf.pivot(index="weekday", columns="hour", values="signins")
                .reindex(order).reindex(columns=range(24)).fillna(0))

    plt.figure(figsize=(14, 5))
    sns.heatmap(pivot, cmap="rocket_r", linewidths=.5, linecolor="#eee",
                cbar_kws={"label": "sign-ins"})
    plt.title(f"Sign-in activity by hour and weekday - last {LOOKBACK_DAYS} days")
    plt.xlabel("Hour of day"); plt.ylabel("")
    plt.tight_layout(); plt.show()

## Why a notebook beats KQL here

KQL renders tables, not heatmaps. Spotting the working-hours block and the lone 3 a.m. Saturday cell is instant visually, and near-impossible to eyeball in a KQL grid.